In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load Data
df = pd.read_csv('online_shoppers_intention.csv')

# 2. Model Preparation
# Define features and target (Excluding 'Month' as requested)
X = df.drop(['Revenue', 'Month'], axis=1)
y = df['Revenue'].astype(int)

# Identify categorical and numerical columns
categorical_cols = ['VisitorType', 'Weekend', 'OperatingSystems', 
                    'Browser', 'Region', 'TrafficType']
numerical_cols = ['Administrative', 'Administrative_Duration', 'Informational', 
                  'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 
                  'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Create the pipeline with Logistic Regression
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# 3. Train and Evaluate
baseline_model.fit(X_train, y_train)

# Predictions
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

# Print Results
print("--- Logistic Regression Baseline Results (Simplified) ---")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 4. Feature Importance (Coefficients)
# Extract feature names from the encoder
cat_feature_names = baseline_model.named_steps['preprocessor']\
    .named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = numerical_cols + list(cat_feature_names)

# Get coefficients
coefficients = baseline_model.named_steps['classifier'].coef_[0]
importance_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
importance_df['Abs_Coefficient'] = importance_df['Coefficient'].abs()
importance_df = importance_df.sort_values(by='Abs_Coefficient', ascending=False)

print("\n--- Top 10 Influential Features ---")
print(importance_df.head(10))

--- Logistic Regression Baseline Results (Simplified) ---
ROC AUC Score: 0.8595

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      2084
           1       0.75      0.34      0.47       382

    accuracy                           0.88      2466
   macro avg       0.82      0.66      0.70      2466
weighted avg       0.87      0.88      0.86      2466


--- Top 10 Influential Features ---
               Feature  Coefficient  Abs_Coefficient
8           PageValues     1.503561         1.503561
34          Browser_12     1.331933         1.331933
29           Browser_7    -0.870327         0.870327
59      TrafficType_15    -0.854091         0.854091
7            ExitRates    -0.760771         0.760771
25           Browser_3    -0.742171         0.742171
20  OperatingSystems_6    -0.650812         0.650812
52       TrafficType_8     0.595607         0.595607
11   VisitorType_Other    -0.544646         0.544646
58